In [ ]:
import random

room = []
m, n = map(int, input("Nhap so dong va cot: ").split())

for i in range(m):
    row = list(map(int, input().split()))
    room.append(row)

k = int(input("Nhap so luong chum k: "))


def printRoomWithAgents(room, current_state_set):
    """In ma trận phòng hiển thị vị trí của tất cả các trạng thái trong chùm k"""

    agent_positions = {(state[0], state[1]) for state in current_state_set}

    for i in range(m):
        for j in range(n):
            if (i, j) in agent_positions:
                print("A", end=" ")
            else:
                print(room[i][j], end=" ")
        print()


def is_clean():
    """Kiểm tra xem toàn bộ phòng đã sạch chưa"""
    for i in range(m):
        for j in range(n):
            if room[i][j] == 1:
                return False
    return True

def evaluate_value(target_x, target_y):
    if room[target_x][target_y] == 1:
        return 1000

    min_dist = float('inf')
    for i in range(m):
        for j in range(n):
            if room[i][j] == 1:
                dist = abs(target_x - i) + abs(target_y - j)
                if dist < min_dist:
                    min_dist = dist

    if min_dist == float('inf'):
        return 0

    return 100 - min_dist


def get_neighbors(x, y):
    """Sinh ra tất cả các trạng thái lân cận của một ô"""
    neighbors = []
    if x > 0: neighbors.append((x - 1, y))
    if y > 0: neighbors.append((x, y - 1))
    if x < m - 1: neighbors.append((x + 1, y))
    if y < n - 1: neighbors.append((x, y + 1))
    return neighbors


Current_State_set = []
for _ in range(k):
    rand_x = random.randint(0, m - 1)
    rand_y = random.randint(0, n - 1)
    Current_State_set.append((rand_x, rand_y))

step = 0
max_steps = 30
found_goal = False

print("\n--- BẮT ĐẦU THUẬT TOÁN LOCAL BEAM SEARCH ---")

while step < max_steps:
    print(f"\n================ STEP {step} ================")
    print(f"Danh sách {k} trạng thái hiện tại trong chùm: {Current_State_set}")

    for state in Current_State_set:
        curr_x, curr_y = state[0], state[1]
        if room[curr_x][curr_y] == 1:
            print(f"-> Trạng thái ({curr_x}, {curr_y}) phát hiện bụi bẩn! Đang dọn...")
            room[curr_x][curr_y] = 0

    printRoomWithAgents(room, Current_State_set)
    print()

    if is_clean():
        print("-> ĐẠT MỤC TIÊU: Toàn bộ phòng đã được dọn sạch hoàn toàn!")
        found_goal = True
        break

    Neighbor_States = set()

    for state in Current_State_set:
        neighbors = get_neighbors(state[0], state[1])
        for n_state in neighbors:
            Neighbor_States.add(n_state)

    Neighbor_States = list(Neighbor_States)
    print(f"-> Sinh ra tổng cộng {len(Neighbor_States)} trạng thái lân cận.")

    goal_neighbor = None
    for neighbor in Neighbor_States:
        if room[neighbor[0]][neighbor[1]] == 1:
            goal_neighbor = neighbor
            break

    if goal_neighbor:
        print(f"-> TRẢ VỀ {goal_neighbor}: Tìm thấy ô bẩn mục tiêu, dừng bước quét!")
        Current_State_set = [goal_neighbor] * k

    else:
        Neighbor_States.sort(key=lambda s: evaluate_value(s[0], s[1]), reverse=True)

        Current_State_set = Neighbor_States[:k]
        print(f"-> Lựa chọn chùm: Đã lọc lấy {k} trạng thái tốt nhất cho bước tiếp theo.")

    step += 1

print("\n--- KẾT QUẢ CUỐI CÙNG ---")
if is_clean() or found_goal:
    print("Thành công: Phòng đã sạch bóng!")
else:
    print("Kết thúc: Dừng thuật toán (Đã đạt giới hạn bước đi hoặc các chùm bị kẹt cục bộ cùng nhau).")